# Introduction to PyTorch Tensors

PyTorch is a deep learning framework built around **tensors** — multi-dimensional arrays similar to NumPy's `ndarray`, but with two key additions:
- **GPU acceleration**: tensors can be moved to a GPU for fast parallel computation.
- **Automatic differentiation**: PyTorch can automatically compute gradients, which is essential for training neural networks.

In this notebook we cover the basics: creating tensors, inspecting their shape, performing operations, and understanding broadcasting.

In [32]:
%pip install torch

Note: you may need to restart the kernel to use updated packages.


In [33]:
import torch

---
## 1. Creating Tensors

The most basic way to create a tensor is `torch.empty(rows, cols)`, which allocates memory **without initializing** it (the values are whatever was in memory). Other common constructors include:
- `torch.zeros(...)` — filled with zeros
- `torch.ones(...)` — filled with ones
- `torch.full(size, value)` — filled with a constant
- `torch.randn(...)` — filled with random values from a standard normal distribution
- `torch.tensor([...])` — from a Python list or NumPy array

In [34]:
x = torch.empty(6, 5)

In [35]:
x.size(), x.shape

(torch.Size([6, 5]), torch.Size([6, 5]))

Every tensor has a **shape** (also accessible via `.size()`), which describes its dimensions. Here we have a 6×5 matrix.

In [36]:
x

tensor([[0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.]])

In [37]:
x.fill_(1.125)

tensor([[1.1250, 1.1250, 1.1250, 1.1250, 1.1250],
        [1.1250, 1.1250, 1.1250, 1.1250, 1.1250],
        [1.1250, 1.1250, 1.1250, 1.1250, 1.1250],
        [1.1250, 1.1250, 1.1250, 1.1250, 1.1250],
        [1.1250, 1.1250, 1.1250, 1.1250, 1.1250],
        [1.1250, 1.1250, 1.1250, 1.1250, 1.1250]])

We can fill a tensor with a constant value using `.fill_()`. Note the trailing underscore — this is a PyTorch convention.

**In-place operations** are suffixed with an underscore (`_`). They modify the tensor directly instead of returning a new one. Examples: `.fill_()`, `.add_()`, `.mul_()`, `.normal_()`, `.sub_()`, `.div_()`.

---
## 2. Indexing and Aggregation

You can index and slice tensors just like NumPy arrays. Use `.item()` to extract a single Python number from a scalar tensor.

In [38]:
x[0,2].item()

1.125

In [39]:
x[:, :2]

tensor([[1.1250, 1.1250],
        [1.1250, 1.1250],
        [1.1250, 1.1250],
        [1.1250, 1.1250],
        [1.1250, 1.1250],
        [1.1250, 1.1250]])

Slicing works like NumPy: `x[:, :2]` selects all rows and the first 2 columns.

In [40]:
x.mean(dim=0)

tensor([1.1250, 1.1250, 1.1250, 1.1250, 1.1250])

**Aggregation functions** like `.mean()`, `.std()`, `.sum()` can operate over the entire tensor or along a specific dimension (`dim`).
- `dim=0` → aggregate across rows (result has one value per column)
- `dim=1` → aggregate across columns (result has one value per row)
- `dim=-2` is equivalent to `dim=0` for a 2D tensor

In [41]:
x.std()

tensor(0.)

In [42]:
x.sum(dim=-2)

tensor([6.7500, 6.7500, 6.7500, 6.7500, 6.7500])

In [43]:
x.sum().item()

33.75

---
## 3. Arithmetic and Linear Algebra

Standard arithmetic operators (`+`, `-`, `*`, `/`, `**`) work **element-wise** on tensors. For matrix operations, use:
- `@` or `torch.mm()` — matrix multiplication
- `.mv()` — matrix-vector product
- `torch.linalg.lstsq()` — least-squares solution to a linear system

In [44]:
x = torch.tensor([ 10., 20., 30.])
y = torch.tensor([ 11., 21., 31.])

In [45]:
x + y

tensor([21., 41., 61.])

`+` is element-wise addition:

In [46]:
x * y

tensor([110., 420., 930.])

`*` is element-wise multiplication (Hadamard product), **not** matrix multiplication:

In [47]:
x**2

tensor([100., 400., 900.])

In [48]:
m = torch.tensor([[ 0., 0., 3. ], [ 0., 2., 0. ], [ 1., 0., 0. ]])
m.mv(x)

tensor([90., 40., 10.])

`.mv()` computes a matrix-vector product. The `@` operator can also be used — it handles both matrix-matrix and matrix-vector cases:

In [49]:
m @ x

tensor([90., 40., 10.])

In [50]:
y = torch.randn(3)
y

tensor([-0.3178, -0.4279, -0.5007])

`torch.linalg.lstsq(A, b)` solves the system $Ax = b$ in the least-squares sense — finding the $x$ that minimises $\|Ax - b\|^2$. When $A$ is square and invertible, this gives the exact solution:

In [51]:
m = torch.randn(3, 3)
q = torch.linalg.lstsq(m, y).solution
m@q

tensor([-0.3178, -0.4279, -0.5007])

---
## 4. Broadcasting

**Broadcasting** allows PyTorch to perform operations on tensors of different shapes by automatically expanding the smaller tensor. The rules are the same as NumPy:

1. If the tensors have different numbers of dimensions, the shape of the smaller tensor is padded with ones on the left.
2. Dimensions of size 1 are stretched to match the other tensor.
3. If two dimensions differ and neither is 1, an error is raised.

**Example:** subtracting a vector of shape `(4,)` from a matrix of shape `(100, 4)` — the vector is broadcast across all 100 rows.

In [52]:
x = torch.empty(100, 4).normal_(2)
x.mean(0)

tensor([2.0546, 2.1608, 2.1558, 1.9794])

In [53]:
x -= x.mean(0)  # Broadcasting: (100,4) - (4,) works!
x.mean(0)  # Should now be ~0 for each column

tensor([1.1444e-07, 1.4305e-07, 2.0981e-07, 1.8835e-07])

---
## 5. Reshaping Tensors

Tensors can be reshaped without copying data using `.view()` or `.reshape()`. The total number of elements must stay the same.

| Method | Description |
|--------|-------------|
| `.view(shape)` | Reshape (requires contiguous memory) |
| `.reshape(shape)` | Reshape (works even if not contiguous) |
| `.unsqueeze(dim)` | Add a dimension of size 1 at position `dim` |
| `.squeeze()` | Remove all dimensions of size 1 |
| `.t()` | Transpose a 2D tensor |
| `.permute(dims)` | Reorder dimensions arbitrarily |

Use `-1` for one dimension to let PyTorch infer it automatically.

In [54]:
a = torch.arange(12)
print("Original:", a.shape, a)

# Reshape into a 3x4 matrix
b = a.view(3, 4)
print("\nview(3, 4):\n", b)

# Use -1 to infer one dimension
c = a.view(-1, 6)
print("\nview(-1, 6):\n", c)

# unsqueeze adds a dimension
d = a.unsqueeze(0)     # shape: (1, 12) — row vector
e = a.unsqueeze(1)     # shape: (12, 1) — column vector
print("\nunsqueeze(0):", d.shape)
print("unsqueeze(1):", e.shape)

Original: torch.Size([12]) tensor([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11])

view(3, 4):
 tensor([[ 0,  1,  2,  3],
        [ 4,  5,  6,  7],
        [ 8,  9, 10, 11]])

view(-1, 6):
 tensor([[ 0,  1,  2,  3,  4,  5],
        [ 6,  7,  8,  9, 10, 11]])

unsqueeze(0): torch.Size([1, 12])
unsqueeze(1): torch.Size([12, 1])


---
## 6. Automatic Differentiation (`autograd`)

PyTorch can automatically compute gradients. This is the foundation of training neural networks: instead of deriving gradients by hand (as we did for linear and logistic regression), PyTorch builds a **computational graph** while executing operations and uses it to compute gradients via the **chain rule** (backpropagation).

To enable gradient tracking on a tensor, set `requires_grad=True`.

In [55]:
# Example: compute the gradient of f(x) = x^2 + 3x at x = 2
# The derivative is f'(x) = 2x + 3, so f'(2) = 7

x = torch.tensor(2.0, requires_grad=True)

# Forward pass: compute f(x)
f = x**2 + 3*x

# Backward pass: compute gradients
f.backward()

print(f"f(2) = {f.item()}")
print(f"f'(2) = {x.grad.item()}")  # Should be 7.0

f(2) = 10.0
f'(2) = 7.0


This works for vectors and matrices too. Here is a more realistic example — computing the gradient of a loss function with respect to a weight vector:

In [56]:
# Linear regression gradient via autograd
# Model: y_hat = X @ w, Loss: MSE = mean((y_hat - y)^2)

torch.manual_seed(42)
X = torch.randn(50, 3)
y = X @ torch.tensor([1.0, -2.0, 0.5]) + 0.1 * torch.randn(50)

w = torch.randn(3, requires_grad=True)

# Forward pass
y_hat = X @ w
loss = ((y_hat - y) ** 2).mean()

# Backward pass
loss.backward()

print(f"Loss: {loss.item():.4f}")
print(f"Gradient of loss w.r.t. w: {w.grad}")

Loss: 3.4767
Gradient of loss w.r.t. w: tensor([-0.6601,  3.2062,  1.1189])


**Key takeaway:** You never need to derive gradients by hand in PyTorch. Define the forward computation, call `.backward()`, and the gradients appear in `.grad`. This is what `torch.nn` and optimizers build on top of.